# Radiosonde/Dropsonde Zarr Pipeline
Demonstration of Zarr pipeline using M2HATS data, written for users to understand the process. This will be put into .py files and done behind the scenes. 

---

# Import required packages

In [1]:
from pathlib import Path
import glob
import xarray as xr
import zarr

---

# Process data

### Create a Zarr store
I created a Zarr store in my scratch directory to contain the individual radiosonde datasets. 

In [2]:
zarr_path = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"
root = zarr.open_group(zarr_path, mode = "a")

### Load data
This M2HATS radiosonde dataset was downloaded from the EOL Field Data Archive as 122 netcdf files, then stored in my scratch directory. I only downloaded ascending files for this case study. 

In [6]:
radiosonde_path = Path("/lustre/desc1/scratch/myasears/sounding_data/m2hats")
radiosonde_files = sorted([p for p in radiosonde_path.iterdir() if p.suffix == ".nc"])

### Populate the Zarr store
I iterate through every netcdf file, open it with xarray, drop incompatible variables, and convert them to Zarr.

In [7]:
for sonde in radiosonde_files:
    sounding_id = sonde.stem
    ds = xr.open_dataset(sonde)
    ds = ds.drop_vars("trajectory") # Drop "trajectory" - data type does not work with Zarr V3
    ds.to_zarr(zarr_path, group=sounding_id, consolidated=False)

---

# Open Zarr
See whether the Zarr store has been populated -- it has.

In [3]:
m2hats_zarr = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"

In [4]:
root = zarr.open_group(m2hats_zarr, mode="r")
sounding_ids = list(root.group_keys())